#### Importing Required Libraries

In [231]:
# Importing libraries for data handling.

import pandas as pd
import numpy as np


# Importing libraries for data visualization.

import plotly.express as px
import plotly.graph_objects as go


# Importing libraries for data preprocessing and model development.

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Importing regression models for demand prediction.

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)


# Importing libraries for model evaluation.

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#### Loading the Engineered Dataset

In [232]:
# Loading the engineered dataset for model development.

df = pd.read_csv(
    '../data/processed/engineered_data.csv'
)

In [233]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Weekend,Inventory_to_Sales_Ratio,Inventory_Gap,Price_Difference,Price_Difference_Percentage,Promotion_Discount,Previous_Demand,Previous_Units_Sold,Rolling_7_Day_Demand,Rolling_7_Day_Sales
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1.911765,93,-13.01,-15.175551,0,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,1,1.309859,22,-8.03,-10.901439,0,115.0,102.0,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,0,1.929577,132,-12.18,-15.087328,15,84.0,71.0,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,0,3.142857,90,6.78,12.354227,0,132.0,142.0,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,0,2.472868,190,2.22,3.871643,25,67.0,42.0,NaN,NaN


In [234]:
# Checking the dataset structure and data types.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         76000 non-null  object 
 1   Store ID                     76000 non-null  object 
 2   Product ID                   76000 non-null  object 
 3   Category                     76000 non-null  object 
 4   Region                       76000 non-null  object 
 5   Inventory Level              76000 non-null  int64  
 6   Units Sold                   76000 non-null  int64  
 7   Units Ordered                76000 non-null  int64  
 8   Price                        76000 non-null  float64
 9   Discount                     76000 non-null  int64  
 10  Weather Condition            76000 non-null  object 
 11  Promotion                    76000 non-null  int64  
 12  Competitor Pricing           76000 non-null  float64
 13  Seasonality     

##### Converting the Date Column

In [235]:
# Converting the Date column into datetime format.

df['Date'] = pd.to_datetime(
    df['Date']
)

#### Sorting the Dataset by Time

In [236]:
# Sorting the dataset by date to maintain chronological order.

df = df.sort_values(
    'Date'
).reset_index(
    drop=True
)

#### Checking Missing Values

In [237]:
# Checking missing values before preparing the modeling dataset.

df.isnull().sum().sort_values(
    ascending=False
)

Rolling_7_Day_Sales            700
Rolling_7_Day_Demand           700
Previous_Units_Sold            100
Previous_Demand                100
Date                             0
Store ID                         0
Product ID                       0
Units Ordered                    0
Price                            0
Discount                         0
Weather Condition                0
Category                         0
Region                           0
Inventory Level                  0
Units Sold                       0
Epidemic                         0
Seasonality                      0
Competitor Pricing               0
Promotion                        0
Demand                           0
Year                             0
Month                            0
Week                             0
Inventory_to_Sales_Ratio         0
Is_Weekend                       0
Day_of_Week                      0
Day                              0
Promotion_Discount               0
Price_Difference_Per

#### Handling the missing values

In [238]:
# Removing rows where lag and rolling features cannot be calculated due to insufficient historical data.

df = df.dropna(
    subset=[
        'Previous_Demand',
        'Previous_Units_Sold',
        'Rolling_7_Day_Demand',
        'Rolling_7_Day_Sales'
    ]
).reset_index(
    drop=True
)

In [239]:
# Verifying that missing values have been removed from the modeling dataset.

df.isnull().sum().sort_values(
    ascending=False
)

Date                           0
Store ID                       0
Product ID                     0
Category                       0
Region                         0
Inventory Level                0
Units Sold                     0
Units Ordered                  0
Price                          0
Discount                       0
Weather Condition              0
Promotion                      0
Competitor Pricing             0
Seasonality                    0
Epidemic                       0
Demand                         0
Year                           0
Month                          0
Week                           0
Day                            0
Day_of_Week                    0
Is_Weekend                     0
Inventory_to_Sales_Ratio       0
Inventory_Gap                  0
Price_Difference               0
Price_Difference_Percentage    0
Promotion_Discount             0
Previous_Demand                0
Previous_Units_Sold            0
Rolling_7_Day_Demand           0
Rolling_7_

In [240]:
# Checking the dataset shape after removing rows with insufficient historical data.

print("Dataset Shape:", df.shape)

Dataset Shape: (75300, 31)


#### Defining Target and Features

In [241]:
# Defining Demand as the target variable for the forecasting models.
target = 'Demand'

# Defining the features that will be used to predict demand.
feature_columns = [
    'Store ID',
    'Product ID',
    'Category',
    'Region',
    'Inventory Level',
    'Units Sold',
    'Units Ordered',
    'Price',
    'Discount',
    'Weather Condition',
    'Promotion',
    'Competitor Pricing',
    'Seasonality',
    'Epidemic',
    'Year',
    'Month',
    'Week',
    'Inventory_to_Sales_Ratio',
    'Is_Weekend',
    'Day_of_Week',
    'Day',
    'Promotion_Discount',
    'Price_Difference_Percentage',
    'Price_Difference',
    'Inventory_Gap',
    'Previous_Demand',
    'Previous_Units_Sold',
    'Rolling_7_Day_Demand',
    'Rolling_7_Day_Sales'
]

# Creating the feature matrix and target variable for model development.

X = df[feature_columns]
y = df[target]

# Verifying the dimensions of the feature matrix and target variable.

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (75300, 29)
Target Shape: (75300,)


#### Checking Feature Leakage

Before training, we're checking whether any feature contains information that would not actually be available when predicting future demand.

###### Checking Feature Availability

In [242]:
# Displaying all features that are currently being considered for model development.

X.columns.tolist()

['Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Sold',
 'Units Ordered',
 'Price',
 'Discount',
 'Weather Condition',
 'Promotion',
 'Competitor Pricing',
 'Seasonality',
 'Epidemic',
 'Year',
 'Month',
 'Week',
 'Inventory_to_Sales_Ratio',
 'Is_Weekend',
 'Day_of_Week',
 'Day',
 'Promotion_Discount',
 'Price_Difference_Percentage',
 'Price_Difference',
 'Inventory_Gap',
 'Previous_Demand',
 'Previous_Units_Sold',
 'Rolling_7_Day_Demand',
 'Rolling_7_Day_Sales']

##### Important point

For this dataset, Demand is our target:

- Demand → What we are trying to predict

Our lagged features are safe:

- Previous_Demand
- Previous_Units_Sold
- Rolling_7_Day_Demand
- Rolling_7_Day_Sales
because they are calculated using previous days only.

However, we need to think carefully about:

- Units Sold
- Units Ordered
- Inventory Level
These values may or may not be available depending on when the prediction is being made.

For a realistic retail system, we can frame the prediction as:

- Predicting demand for a day using information available at the beginning of that day.

Under that framing, today's inventory, price, promotion, weather forecast, etc. can potentially be available.

But today's Units Sold cannot be used, because sales are the result of today's demand.

So we should remove Units Sold from the model features.

##### Removing Target-Leaking Features

In [243]:
# Removing current-day Units Sold because it is influenced by the demand being predicted.

feature_columns = [
    col for col in feature_columns
    if col != 'Units Sold'
]

In [244]:
# Recreating the feature matrix after removing the current-day sales feature.

X = df[feature_columns]

y = df[target]

In [245]:
# Displaying the final features that will be used for model development.

X.columns.tolist()

['Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Ordered',
 'Price',
 'Discount',
 'Weather Condition',
 'Promotion',
 'Competitor Pricing',
 'Seasonality',
 'Epidemic',
 'Year',
 'Month',
 'Week',
 'Inventory_to_Sales_Ratio',
 'Is_Weekend',
 'Day_of_Week',
 'Day',
 'Promotion_Discount',
 'Price_Difference_Percentage',
 'Price_Difference',
 'Inventory_Gap',
 'Previous_Demand',
 'Previous_Units_Sold',
 'Rolling_7_Day_Demand',
 'Rolling_7_Day_Sales']

#### Creating a Time-Based Train-Test Split

This is very important for our project.

We should not randomly split this dataset.

Why?

Imagine:

- 2022 → Training data
- 2023 → Training data
- 2024 → Testing data

This is realistic.

The model learns from the past and is tested on the future.

If we randomly shuffle the data, the model could see information from 2024 while trying to predict 2022, which isn't how real forecasting works.

#### Checking the Date Range

In [246]:
# Checking the available date range before creating the time-based split.

print("Minimum Date:", df['Date'].min())
print("Maximum Date:", df['Date'].max())

Minimum Date: 2022-01-08 00:00:00
Maximum Date: 2024-01-30 00:00:00


#### Defining Split Date

We have data from 2022-01-01 to 2024-01-30.

We'll use the last part of the timeline as our test set.

In [247]:
# Defining the date boundary for separating historical training data from future testing data.

split_date = pd.Timestamp('2023-10-01')

#### Creating Training and Testing Sets

In [248]:
# Creating the training dataset using observations before the split date.

train_data = df[
    df['Date'] < split_date
].copy()


# Creating the testing dataset using observations on and after the split date.

test_data = df[
    df['Date'] >= split_date
].copy()

#### Creating X and y for Training

In [249]:
# Creating the training features and target variable.

X_train = train_data[feature_columns]

y_train = train_data[target]

#### Creating X and y for Testing

In [250]:
# Creating the testing features and target variable.

X_test = test_data[feature_columns]

y_test = test_data[target]

#### Checking Split Sizes

In [251]:
# Checking the number of observations in the training and testing datasets.

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (63100, 28)
Testing Data Shape: (12200, 28)


#### Checking Split Dates

In [252]:
# Checking the date ranges of the training and testing datasets.

print(
    "Training Date Range:",
    train_data['Date'].min(),
    "to",
    train_data['Date'].max()
)

print(
    "Testing Date Range:",
    test_data['Date'].min(),
    "to",
    test_data['Date'].max()
)

Training Date Range: 2022-01-08 00:00:00 to 2023-09-30 00:00:00
Testing Date Range: 2023-10-01 00:00:00 to 2024-01-30 00:00:00


Our data is now conceptually:


                 TIME
                  ↓

2022 ─────────────────────── 2023-09-30 | 2023-10-01 ─────── 2024
       TRAINING DATA                     |       TEST DATA
                                        ↑
                                   Split Date



The model will learn from the past and then we'll ask it to predict the future.



#### Encoding Categorical Features

##### Defining Categorical Features

In [253]:
# Defining the categorical features that will be encoded before model training.

categorical_features = [
    'Store ID',
    'Product ID',
    'Category',
    'Region',
    'Weather Condition',
    'Seasonality'
]

##### Defining Numerical Features

In [254]:
# Defining the numerical features that will be passed directly to the models.

numerical_features = [
    col for col in feature_columns
    if col not in categorical_features
]

##### Displaying Feature Groups

In [255]:
# Displaying the categorical and numerical feature groups for verification.

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']

Numerical Features:
['Inventory Level', 'Units Ordered', 'Price', 'Discount', 'Promotion', 'Competitor Pricing', 'Epidemic', 'Year', 'Month', 'Week', 'Inventory_to_Sales_Ratio', 'Is_Weekend', 'Day_of_Week', 'Day', 'Promotion_Discount', 'Price_Difference_Percentage', 'Price_Difference', 'Inventory_Gap', 'Previous_Demand', 'Previous_Units_Sold', 'Rolling_7_Day_Demand', 'Rolling_7_Day_Sales']


#### Creating the Preprocessing Pipeline

In [256]:
# Creating a one-hot encoder for converting categorical features into numerical features.

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

##### Creating the Preprocessing Transformer

In [257]:
# Creating a preprocessing transformer for encoding categorical features while retaining numerical features.

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            encoder,
            categorical_features
        ),
        (
            'numerical',
            'passthrough',
            numerical_features
        )
    ]
)

#### Checking the Preprocessing Pipeline

Before training a model, let's make sure the transformation works.

##### Fitting the Preprocessor

In [258]:
# Fitting the preprocessing transformer using only the training data.

preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

##### Transforming Training Data

In [259]:
# Transforming the training features using the fitted preprocessing pipeline.

X_train_processed = preprocessor.transform(
    X_train
)

##### Transforming Testing Data

In [260]:
# Transforming the testing features using the preprocessing pipeline fitted on the training data.

X_test_processed = preprocessor.transform(
    X_test
)

##### Checking Processed Data Shape

In [261]:
# Checking the dimensions of the processed training and testing feature matrices.

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (63100, 64)
Processed Testing Shape: (12200, 64)


#### Establishing a Baseline Model

For a demand forecasting problem, a very useful baseline is:

- Predicting the average historical demand from the training data for every future observation.

It's deliberately simple.

If our baseline MAE is 45 and a Random Forest gives MAE 20, we know the model is providing meaningful improvement.

##### Calculating Baseline Prediction

In [262]:
# Calculating the average demand from the training data for the baseline prediction.

baseline_prediction = y_train.mean()

##### Creating Baseline Predictions

In [263]:
# Creating baseline predictions using the average training demand.

y_baseline_pred = np.full(
    shape=len(y_test),
    fill_value=baseline_prediction
)

##### Evaluating Baseline MAE

In [264]:
# Calculating the Mean Absolute Error for the baseline predictions.

baseline_mae = mean_absolute_error(
    y_test,
    y_baseline_pred
)

print("Baseline MAE:", baseline_mae)

Baseline MAE: 36.491772642955496


##### Evaluating Baseline RMSE

In [265]:
# Calculating the Root Mean Squared Error for the baseline predictions.

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_baseline_pred
    )
)

print("Baseline RMSE:", baseline_rmse)

Baseline RMSE: 45.41942310451735


##### Evaluating Baseline R²

In [266]:
# Calculating the R² score for the baseline predictions.

baseline_r2 = r2_score(
    y_test,
    y_baseline_pred
)

print("Baseline R²:", baseline_r2)

Baseline R²: -0.021844943419516483


##### Creating a Baseline Results Table

In [267]:
# Creating a results table for storing baseline model performance.

model_results = pd.DataFrame({
    'Model': ['Baseline'],
    'MAE': [baseline_mae],
    'RMSE': [baseline_rmse],
    'R2': [baseline_r2]
})

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845


#### Training Linear Regression

In [268]:
# Importing the Linear Regression algorithm for demand prediction.

from sklearn.linear_model import LinearRegression

In [269]:
# Creating and training the Linear Regression model using the processed training data.

linear_regression = LinearRegression()

linear_regression.fit(
    X_train_processed,
    y_train
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](64,)","[-0.07,-0.37,-0.1 ,..., 0.02, 0.2 ,-0.06]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-272.1
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,64
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,54
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](64,)","[80043.01,41146.19,17790. ,..., 0. , 0. , 0. ]"


In [270]:
# Predicting demand values using the processed test data.

y_pred_lr = linear_regression.predict(
    X_test_processed
)

##### Evaluating Linear Regression

In [271]:
# Calculating the Mean Absolute Error for the Linear Regression model.

lr_mae = mean_absolute_error(
    y_test,
    y_pred_lr
)

# Calculating the Root Mean Squared Error for the Linear Regression model.

lr_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_lr
    )
)


# Calculating the R² score for the Linear Regression model.

lr_r2 = r2_score(
    y_test,
    y_pred_lr
)



print("Linear Regression MAE:", lr_mae)
print("Linear Regression RMSE:", lr_rmse)
print("Linear Regression R²:", lr_r2)


Linear Regression MAE: 15.154548815914227
Linear Regression RMSE: 20.365632720086932
Linear Regression R²: 0.7945540485792104


##### Adding Linear Regression to the Results Table

In [272]:
# Adding the Linear Regression performance metrics to the model comparison table.

linear_regression_results = pd.DataFrame({
    'Model': ['Linear Regression'],
    'MAE': [lr_mae],
    'RMSE': [lr_rmse],
    'R2': [lr_r2]
})

model_results = pd.concat(
    [
        model_results,
        linear_regression_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554


#### Training Ridge Regression

In [273]:
# Importing the Ridge Regression algorithm for demand prediction.

from sklearn.linear_model import Ridge

In [274]:
# Creating and training the Ridge Regression model using the processed training data.

ridge_regression = Ridge(
    alpha=1.0
)

ridge_regression.fit(
    X_train_processed,
    y_train
)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [275]:
# Predicting demand values using the Ridge Regression model.

y_pred_ridge = ridge_regression.predict(
    X_test_processed
)

##### Evaluating Ridge Regression

In [276]:
# Calculating the evaluation metrics for the Ridge Regression model.

ridge_mae = mean_absolute_error(
    y_test,
    y_pred_ridge
)

ridge_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_ridge
    )
)

ridge_r2 = r2_score(
    y_test,
    y_pred_ridge
)

print("Ridge Regression MAE:", ridge_mae)
print("Ridge Regression RMSE:", ridge_rmse)
print("Ridge Regression R²:", ridge_r2)

Ridge Regression MAE: 15.154426048175662
Ridge Regression RMSE: 20.365598700412626
Ridge Regression R²: 0.7945547349510618


In [277]:
# Adding the Ridge Regression performance metrics to the model comparison table.

ridge_results = pd.DataFrame({
    'Model': ['Ridge Regression'],
    'MAE': [ridge_mae],
    'RMSE': [ridge_rmse],
    'R2': [ridge_r2]
})

model_results = pd.concat(
    [
        model_results,
        ridge_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555


#### Training Lasso Regression

In [278]:
# Importing the Lasso Regression algorithm for demand prediction.

from sklearn.linear_model import Lasso

In [279]:
# Creating and training the Lasso Regression model using the processed training data.

lasso_regression = Lasso(
    alpha=1.0,
    max_iter=10000
)

lasso_regression.fit(
    X_train_processed,
    y_train
)

,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [280]:
# Predicting demand values using the Lasso Regression model.

y_pred_lasso = lasso_regression.predict(
    X_test_processed
)

##### Evaluating Lasso Regression

In [281]:
# Calculating the evaluation metrics for the Lasso Regression model.

lasso_mae = mean_absolute_error(
    y_test,
    y_pred_lasso
)

lasso_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_lasso
    )
)

lasso_r2 = r2_score(
    y_test,
    y_pred_lasso
)

print("Lasso Regression MAE:", lasso_mae)
print("Lasso Regression RMSE:", lasso_rmse)
print("Lasso Regression R²:", lasso_r2)

Lasso Regression MAE: 15.878327610277013
Lasso Regression RMSE: 21.387819003009295
Lasso Regression R²: 0.7734131128697799


In [282]:
# Adding the Lasso Regression performance metrics to the model comparison table.

lasso_results = pd.DataFrame({
    'Model': ['Lasso Regression'],
    'MAE': [lasso_mae],
    'RMSE': [lasso_rmse],
    'R2': [lasso_r2]
})

model_results = pd.concat(
    [
        model_results,
        lasso_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413


#### Training Decision Tree Regression

In [283]:
# Importing the Decision Tree Regression algorithm for demand prediction.

from sklearn.tree import DecisionTreeRegressor

In [284]:
# Creating and training the Decision Tree Regression model using the processed training data.

decision_tree = DecisionTreeRegressor(
    random_state=42
)

decision_tree.fit(
    X_train_processed,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes 

In [285]:
# Predicting demand values using the trained Decision Tree Regression model.

y_pred_decision_tree = decision_tree.predict(
    X_test_processed
)

##### Evaluating Decision Tree Regression

In [286]:
# Calculating the evaluation metrics for the Decision Tree Regression model.

decision_tree_mae = mean_absolute_error(
    y_test,
    y_pred_decision_tree
)

decision_tree_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_decision_tree
    )
)

decision_tree_r2 = r2_score(
    y_test,
    y_pred_decision_tree
)

print("Decision Tree MAE:", decision_tree_mae)
print("Decision Tree RMSE:", decision_tree_rmse)
print("Decision Tree R²:", decision_tree_r2)

Decision Tree MAE: 17.284016393442624
Decision Tree RMSE: 23.607679703511877
Decision Tree R²: 0.7239368868181748


In [287]:
# Adding the Decision Tree Regression performance metrics to the model comparison table.

decision_tree_results = pd.DataFrame({
    'Model': ['Decision Tree Regression'],
    'MAE': [decision_tree_mae],
    'RMSE': [decision_tree_rmse],
    'R2': [decision_tree_r2]
})

model_results = pd.concat(
    [
        model_results,
        decision_tree_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937


#### Training Random Forest Regression

In [288]:
# Importing the Random Forest Regression algorithm for demand prediction.

from sklearn.ensemble import RandomForestRegressor

In [289]:
# Creating and training the Random Forest Regression model using the processed training data.

random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train_processed,
    y_train
)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [290]:
# Predicting demand values using the trained Random Forest Regression model.

y_pred_random_forest = random_forest.predict(
    X_test_processed
)

##### Evaluating Random Forest Regression

In [291]:
# Calculating the evaluation metrics for the Random Forest Regression model.

random_forest_mae = mean_absolute_error(
    y_test,
    y_pred_random_forest
)

random_forest_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_random_forest
    )
)

random_forest_r2 = r2_score(
    y_test,
    y_pred_random_forest
)

print("Random Forest MAE:", random_forest_mae)
print("Random Forest RMSE:", random_forest_rmse)
print("Random Forest R²:", random_forest_r2)

Random Forest MAE: 12.2185631147541
Random Forest RMSE: 16.09470964119658
Random Forest R²: 0.8716877664053964


In [292]:
# Adding the Random Forest Regression performance metrics to the model comparison table.

random_forest_results = pd.DataFrame({
    'Model': ['Random Forest Regression'],
    'MAE': [random_forest_mae],
    'RMSE': [random_forest_rmse],
    'R2': [random_forest_r2]
})

model_results = pd.concat(
    [
        model_results,
        random_forest_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688


#### Training Gradient Boosting Regression

In [293]:
# Importing the Gradient Boosting Regression algorithm for demand prediction.

from sklearn.ensemble import GradientBoostingRegressor

In [294]:
# Creating and training the Gradient Boosting Regression model using the processed training data.

gradient_boosting = GradientBoostingRegressor(
    random_state=42
)

gradient_boosting.fit(
    X_train_processed,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [295]:
# Predicting demand values using the trained Gradient Boosting Regression model.

y_pred_gradient_boosting = gradient_boosting.predict(
    X_test_processed
)

##### Evaluating Gradient Boosting Regression

In [296]:
# Calculating the evaluation metrics for the Gradient Boosting Regression model.

gradient_boosting_mae = mean_absolute_error(
    y_test,
    y_pred_gradient_boosting
)

gradient_boosting_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_gradient_boosting
    )
)

gradient_boosting_r2 = r2_score(
    y_test,
    y_pred_gradient_boosting
)

print("Gradient Boosting MAE:", gradient_boosting_mae)
print("Gradient Boosting RMSE:", gradient_boosting_rmse)
print("Gradient Boosting R²:", gradient_boosting_r2)

Gradient Boosting MAE: 13.429953244589187
Gradient Boosting RMSE: 17.480096315145726
Gradient Boosting R²: 0.8486475640028063


In [297]:
# Adding the Gradient Boosting Regression performance metrics to the model comparison table.

gradient_boosting_results = pd.DataFrame({
    'Model': ['Gradient Boosting Regression'],
    'MAE': [gradient_boosting_mae],
    'RMSE': [gradient_boosting_rmse],
    'R2': [gradient_boosting_r2]
})

model_results = pd.concat(
    [
        model_results,
        gradient_boosting_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648


#### Training XGBoost Boosting Regression

In [298]:
# Importing the XGBoost Regression algorithm for demand prediction.

from xgboost import XGBRegressor

In [299]:
# Creating and training the XGBoost Regression model using the processed training data.

xgboost_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgboost_model.fit(
    X_train_processed,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [300]:
# Predicting demand values using the trained XGBoost Regression model.

y_pred_xgboost = xgboost_model.predict(
    X_test_processed
)

##### Evaluating XGBoost

In [301]:
# Calculating the evaluation metrics for the XGBoost Regression model.

xgboost_mae = mean_absolute_error(
    y_test,
    y_pred_xgboost
)

xgboost_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgboost
    )
)

xgboost_r2 = r2_score(
    y_test,
    y_pred_xgboost
)

print("XGBoost MAE:", xgboost_mae)
print("XGBoost RMSE:", xgboost_rmse)
print("XGBoost R²:", xgboost_r2)

XGBoost MAE: 11.580911636352539
XGBoost RMSE: 15.131740612851166
XGBoost R²: 0.886582612991333


In [302]:
# Adding the XGBoost Regression performance metrics to the model comparison table.

xgboost_results = pd.DataFrame({
    'Model': ['XGBoost Regression'],
    'MAE': [xgboost_mae],
    'RMSE': [xgboost_rmse],
    'R2': [xgboost_r2]
})

model_results = pd.concat(
    [
        model_results,
        xgboost_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648
7,XGBoost Regression,11.580912,15.131741,0.886583


#### Training Extra Trees Regression

In [303]:
# Importing the Extra Trees Regression algorithm for demand prediction.

from sklearn.ensemble import ExtraTreesRegressor

In [304]:
# Creating and training the Extra Trees Regression model using the processed training data.

extra_trees = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

extra_trees.fit(
    X_train_processed,
    y_train
)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls 3 sources of randomness:- the bootstrapping of the samples used when building trees (if ``bootstrap=True``)- the sampling of the features to consider when looking for the best split at each node (if ``max_features < n_features``)- the draw of the splits for each of the `max_features`See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` f

In [305]:
# Predicting demand values using the trained Extra Trees Regression model.

y_pred_extra_trees = extra_trees.predict(
    X_test_processed
)

##### Evaluating Extra Trees Regression

In [306]:
# Calculating the evaluation metrics for the Extra Trees Regression model.

extra_trees_mae = mean_absolute_error(
    y_test,
    y_pred_extra_trees
)

extra_trees_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_extra_trees
    )
)

extra_trees_r2 = r2_score(
    y_test,
    y_pred_extra_trees
)

print("Extra Trees MAE:", extra_trees_mae)
print("Extra Trees RMSE:", extra_trees_rmse)
print("Extra Trees R²:", extra_trees_r2)

Extra Trees MAE: 10.876563114754099
Extra Trees RMSE: 14.656385487751534
Extra Trees R²: 0.8935965937587478


In [307]:
# Adding the Extra Trees Regression performance metrics to the model comparison table.

extra_trees_results = pd.DataFrame({
    'Model': ['Extra Trees Regression'],
    'MAE': [extra_trees_mae],
    'RMSE': [extra_trees_rmse],
    'R2': [extra_trees_r2]
})

model_results = pd.concat(
    [
        model_results,
        extra_trees_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648
7,XGBoost Regression,11.580912,15.131741,0.886583
8,Extra Trees Regression,10.876563,14.656385,0.893597


#### Training HistGradientBoosting Regression

In [308]:
# Importing the HistGradientBoosting Regression algorithm for demand prediction.

from sklearn.ensemble import HistGradientBoostingRegressor

In [309]:
# Creating and training the HistGradientBoosting Regression model using the processed training data.

hist_gradient_boosting = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_depth=None,
    random_state=42
)

hist_gradient_boosting.fit(
    X_train_processed,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [310]:
# Predicting demand values using the trained HistGradientBoosting Regression model.

y_pred_hist_gradient_boosting = (
    hist_gradient_boosting.predict(
        X_test_processed
    )
)

##### Evaluating HistGradientBoosting Regression

In [311]:
# Calculating the evaluation metrics for the HistGradientBoosting Regression model.

hist_gradient_boosting_mae = mean_absolute_error(
    y_test,
    y_pred_hist_gradient_boosting
)

hist_gradient_boosting_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_hist_gradient_boosting
    )
)

hist_gradient_boosting_r2 = r2_score(
    y_test,
    y_pred_hist_gradient_boosting
)

print(
    "HistGradientBoosting MAE:",
    hist_gradient_boosting_mae
)

print(
    "HistGradientBoosting RMSE:",
    hist_gradient_boosting_rmse
)

print(
    "HistGradientBoosting R²:",
    hist_gradient_boosting_r2
)

HistGradientBoosting MAE: 11.783124837222916
HistGradientBoosting RMSE: 15.291003279520657
HistGradientBoosting R²: 0.8841826201428773


In [312]:
# Adding the HistGradientBoosting Regression performance metrics to the model comparison table.

hist_gradient_boosting_results = pd.DataFrame({
    'Model': ['HistGradientBoosting Regression'],
    'MAE': [hist_gradient_boosting_mae],
    'RMSE': [hist_gradient_boosting_rmse],
    'R2': [hist_gradient_boosting_r2]
})

model_results = pd.concat(
    [
        model_results,
        hist_gradient_boosting_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648
7,XGBoost Regression,11.580912,15.131741,0.886583
8,Extra Trees Regression,10.876563,14.656385,0.893597
9,HistGradientBoosting Regression,11.783125,15.291003,0.884183


#### Creating Voting Regression

In [313]:
# Importing the Voting Regression ensemble algorithm.

from sklearn.ensemble import VotingRegressor

In [314]:
# Creating a Voting Regression ensemble using the three strongest individual models.

voting_regressor = VotingRegressor(
    estimators=[
        ('extra_trees', extra_trees),
        ('xgboost', xgboost_model),
        ('random_forest', random_forest)
    ]
)

In [315]:
# Training the Voting Regression ensemble using the processed training data.

voting_regressor.fit(
    X_train_processed,
    y_train
)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('extra_trees', ...), ('xgboost', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
estimators_ estimators_: list of regressorsThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'.,list,"[ExtraTreesReg...ndom_state=42), XGBRegressor(...ree=None, ...), RandomForestR...ndom_state=42)]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,64
named_estimators_ named_estimators_: :class:`~sklearn.utils.Bunch`Attribute to access any fitted sub-estimators by name... versionadded:: 0.20,Bunch,{'extra_trees...dom_state=42)}
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls 3 sources of randomness:- the bootstrapping of the samples used when building trees (if ``bootstrap=True``)- the sampling of the features to consider when looking for the best split at each node (if ``max_features < n_features``)- the draw of the splits for each of the `max_features`See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100


In [316]:
# Predicting demand values using the Voting Regression ensemble.

y_pred_voting = voting_regressor.predict(
    X_test_processed
)

##### Evaluating Voting Regression

In [317]:
# Calculating the evaluation metrics for the Voting Regression ensemble.

voting_mae = mean_absolute_error(
    y_test,
    y_pred_voting
)

voting_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_voting
    )
)

voting_r2 = r2_score(
    y_test,
    y_pred_voting
)

print("Voting Regression MAE:", voting_mae)
print("Voting Regression RMSE:", voting_rmse)
print("Voting Regression R²:", voting_r2)

Voting Regression MAE: 11.044994023188345
Voting Regression RMSE: 14.545320411859628
Voting Regression R²: 0.895203118850949


In [318]:
# Adding the Voting Regression performance metrics to the model comparison table.

voting_results = pd.DataFrame({
    'Model': ['Voting Regression'],
    'MAE': [voting_mae],
    'RMSE': [voting_rmse],
    'R2': [voting_r2]
})

model_results = pd.concat(
    [
        model_results,
        voting_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648
7,XGBoost Regression,11.580912,15.131741,0.886583
8,Extra Trees Regression,10.876563,14.656385,0.893597
9,HistGradientBoosting Regression,11.783125,15.291003,0.884183


#### Creating Stacking Regression

In [319]:
# Importing the Stacking Regression ensemble algorithm.

from sklearn.ensemble import StackingRegressor

In [320]:
# Creating the Ridge Regression model to combine the predictions from the base models.

stacking_final_estimator = Ridge(
    alpha=1.0
)

In [321]:
# Defining the base models that will provide predictions to the stacking ensemble.

stacking_estimators = [
    ('extra_trees', extra_trees),
    ('xgboost', xgboost_model),
    ('random_forest', random_forest)
]

In [322]:
# Creating the Stacking Regression ensemble using the selected base models and meta-model.

stacking_regressor = StackingRegressor(
    estimators=stacking_estimators,
    final_estimator=stacking_final_estimator,
    cv=5,
    n_jobs=-1
)

In [323]:
# Training the Stacking Regression ensemble using the processed training data.

stacking_regressor.fit(
    X_train_processed,
    y_train
)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('extra_trees', ...), ('xgboost', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Ridge()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[ExtraTreesReg...ndom_state=42), XGBRegressor(...ree=None, ...), RandomForestR...ndom_state=42)]"
final_estimator_ final_estimator_: estimatorThe regressor fit on the output of `estimators_` and responsible forfinal predictions.,Ridge,Ridge()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,64
named_estimators_ named_estimators_: :class:`~sklearn.utils.Bunch`Attribute to access any fitted sub-estimators by name.,Bunch,{'extra_trees...dom_state=42)}


In [324]:
# Predicting demand values using the trained Stacking Regression ensemble.

y_pred_stacking = stacking_regressor.predict(
    X_test_processed
)

##### Evaluating Stacking Regression

In [325]:
# Calculating the evaluation metrics for the Stacking Regression ensemble.

stacking_mae = mean_absolute_error(
    y_test,
    y_pred_stacking
)

stacking_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_stacking
    )
)

stacking_r2 = r2_score(
    y_test,
    y_pred_stacking
)

print("Stacking Regression MAE:", stacking_mae)
print("Stacking Regression RMSE:", stacking_rmse)
print("Stacking Regression R²:", stacking_r2)

Stacking Regression MAE: 10.750887398033603
Stacking Regression RMSE: 14.194631034138608
Stacking Regression R²: 0.9001955308031612


In [326]:
# Adding the Stacking Regression performance metrics to the model comparison table.

stacking_results = pd.DataFrame({
    'Model': ['Stacking Regression'],
    'MAE': [stacking_mae],
    'RMSE': [stacking_rmse],
    'R2': [stacking_r2]
})

model_results = pd.concat(
    [
        model_results,
        stacking_results
    ],
    ignore_index=True
)

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648
7,XGBoost Regression,11.580912,15.131741,0.886583
8,Extra Trees Regression,10.876563,14.656385,0.893597
9,HistGradientBoosting Regression,11.783125,15.291003,0.884183


#### Preparing Hyperparameter Tuning

In [327]:
# Importing RandomizedSearchCV for searching through different hyperparameter combinations.

from sklearn.model_selection import RandomizedSearchCV

##### Defining Extra Trees parameters

In [328]:
# Defining the hyperparameter ranges that will be tested for the Extra Trees model.

extra_trees_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': [1.0, 'sqrt']
}

##### Creating the Extra Trees search

In [329]:
# Creating the randomized hyperparameter search for Extra Trees using five-fold cross-validation.

extra_trees_search = RandomizedSearchCV(
    estimator=ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=extra_trees_param_grid,
    n_iter=5,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

##### Running Extra Trees tuning

In [330]:
# Searching for the Extra Trees hyperparameter combination that provides the best cross-validation performance.

extra_trees_search.fit(
    X_train_processed,
    y_train
)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",ExtraTreesReg...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 20], 'max_features': [1.0, 'sqrt'], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function wh

In [331]:
# Displaying the hyperparameter combination that produced the best cross-validation performance.

print("Best Extra Trees Parameters:")

print(
    extra_trees_search.best_params_
)

Best Extra Trees Parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 20}


##### Displaying the best CV RMSE

In [332]:
# Displaying the best cross-validation RMSE obtained during Extra Trees hyperparameter tuning.

print(
    "Best Cross-Validation RMSE:",
    -extra_trees_search.best_score_
)

Best Cross-Validation RMSE: 16.755412094647962


##### Generating tuned Extra Trees predictions

In [333]:
# Predicting demand on the unseen test data using the best Extra Trees model found during hyperparameter tuning.

tuned_extra_trees = extra_trees_search.best_estimator_

y_pred_tuned_extra_trees = tuned_extra_trees.predict(
    X_test_processed
)

##### Evaluating tuned Extra Trees

In [334]:
# Calculating the evaluation metrics for the tuned Extra Trees Regression model.

tuned_extra_trees_mae = mean_absolute_error(
    y_test,
    y_pred_tuned_extra_trees
)

tuned_extra_trees_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tuned_extra_trees
    )
)

tuned_extra_trees_r2 = r2_score(
    y_test,
    y_pred_tuned_extra_trees
)

print("Tuned Extra Trees MAE:", tuned_extra_trees_mae)
print("Tuned Extra Trees RMSE:", tuned_extra_trees_rmse)
print("Tuned Extra Trees R²:", tuned_extra_trees_r2)

Tuned Extra Trees MAE: 11.018016956060796
Tuned Extra Trees RMSE: 14.809777862570028
Tuned Extra Trees R²: 0.8913577224231624


#### Tuning XGBoost

In [335]:
# Defining a lightweight set of hyperparameter combinations for the XGBoost model.

xgboost_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

In [336]:
# Creating a lightweight randomized hyperparameter search for XGBoost using three-fold cross-validation.

xgboost_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=xgboost_param_grid,
    n_iter=5,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [337]:
# Searching for a strong XGBoost hyperparameter combination using the lightweight search.

xgboost_search.fit(
    X_train_processed,
    y_train
)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5], 'min_child_weight': [1, 3], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function wh

In [338]:
# Displaying the hyperparameter combination that produced the best cross-validation performance.

print("Best XGBoost Parameters:")

print(
    xgboost_search.best_params_
)

Best XGBoost Parameters:
{'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [339]:
# Displaying the best cross-validation RMSE obtained during XGBoost hyperparameter tuning.

print(
    "Best Cross-Validation RMSE:",
    -xgboost_search.best_score_
)

Best Cross-Validation RMSE: 16.153263727823894


In [340]:
# Predicting demand on the unseen test data using the best XGBoost model found during hyperparameter tuning.

tuned_xgboost = xgboost_search.best_estimator_

y_pred_tuned_xgboost = tuned_xgboost.predict(
    X_test_processed
)

In [341]:
# Calculating the evaluation metrics for the tuned XGBoost Regression model.

tuned_xgboost_mae = mean_absolute_error(
    y_test,
    y_pred_tuned_xgboost
)

tuned_xgboost_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tuned_xgboost
    )
)

tuned_xgboost_r2 = r2_score(
    y_test,
    y_pred_tuned_xgboost
)

print("Tuned XGBoost MAE:", tuned_xgboost_mae)
print("Tuned XGBoost RMSE:", tuned_xgboost_rmse)
print("Tuned XGBoost R²:", tuned_xgboost_r2)

Tuned XGBoost MAE: 11.402504920959473
Tuned XGBoost RMSE: 14.89000297920397
Tuned XGBoost R²: 0.8901774883270264


#### Tuning Random Forest

In [342]:
# Defining a lightweight set of hyperparameter combinations for the Random Forest model.

random_forest_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': [1.0, 'sqrt']
}

In [343]:
# Creating a lightweight randomized hyperparameter search for Random Forest using three-fold cross-validation.

random_forest_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=random_forest_param_grid,
    n_iter=5,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [344]:
# Searching for a strong Random Forest hyperparameter combination using the lightweight search.

random_forest_search.fit(
    X_train_processed,
    y_train
)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 20], 'max_features': [1.0, 'sqrt'], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function wh

In [345]:
# Displaying the hyperparameter combination that produced the best cross-validation performance.

print("Best Random Forest Parameters:")

print(
    random_forest_search.best_params_
)

Best Random Forest Parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 20}


In [346]:
# Displaying the hyperparameter combination that produced the best cross-validation performance.

print("Best Random Forest Parameters:")

print(
    random_forest_search.best_params_
)

Best Random Forest Parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 20}


In [347]:
# Predicting demand on the unseen test data using the best Random Forest model found during hyperparameter tuning.

tuned_random_forest = random_forest_search.best_estimator_

y_pred_tuned_random_forest = tuned_random_forest.predict(
    X_test_processed
)

In [348]:
# Calculating the evaluation metrics for the tuned Random Forest Regression model.

tuned_random_forest_mae = mean_absolute_error(
    y_test,
    y_pred_tuned_random_forest
)

tuned_random_forest_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tuned_random_forest
    )
)

tuned_random_forest_r2 = r2_score(
    y_test,
    y_pred_tuned_random_forest
)

print("Tuned Random Forest MAE:", tuned_random_forest_mae)
print("Tuned Random Forest RMSE:", tuned_random_forest_rmse)
print("Tuned Random Forest R²:", tuned_random_forest_r2)

Tuned Random Forest MAE: 12.158306528125674
Tuned Random Forest RMSE: 16.001341968080926
Tuned Random Forest R²: 0.8731721628927598


#### Preparing the Best Base Models

In [349]:
# Defining the best-performing versions of the base models for the final stacking experiment.

best_extra_trees = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

best_xgboost = tuned_xgboost

best_random_forest = tuned_random_forest

#### Creating the Final Stacking Model

In [350]:
# Creating a stacking regression model using the best-performing versions of the base models.

final_stacking = StackingRegressor(
    estimators=[
        ('extra_trees', best_extra_trees),
        ('xgboost', best_xgboost),
        ('random_forest', best_random_forest)
    ],
    final_estimator=Ridge(),
    n_jobs=-1
)

In [351]:
final_estimator=Ridge()

Extra Trees ──────┐
                  │
XGBoost ──────────┼──→ Ridge ──→ Final Demand Prediction
                  │
Random Forest ────┘

##### Fitting the Final Stacking Model

In [352]:
# Fitting the final stacking regression model using the training data.

final_stacking.fit(
    X_train_processed,
    y_train
)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('extra_trees', ...), ('xgboost', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Ridge()
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[ExtraTreesReg...ndom_state=42), XGBRegressor(...ree=None, ...), RandomForestR...ndom_state=42)]"
final_estimator_ final_estimator_: estimatorThe regressor fit on the output of `estimators_` and responsible forfinal predictions.,Ridge,Ridge()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,64
named_estimators_ named_estimators_: :class:`~sklearn.utils.Bunch`Attribute to access any fitted sub-estimators by name.,Bunch,{'extra_trees...dom_state=42)}


In [353]:
# Predicting demand on the unseen test data using the final stacking model.

y_pred_final_stacking = final_stacking.predict(
    X_test_processed
)

##### Evaluating Final Stacking

In [354]:
# Calculating the evaluation metrics for the final stacking regression model.

final_stacking_mae = mean_absolute_error(
    y_test,
    y_pred_final_stacking
)

final_stacking_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_final_stacking
    )
)

final_stacking_r2 = r2_score(
    y_test,
    y_pred_final_stacking
)

print("Final Stacking MAE:", final_stacking_mae)
print("Final Stacking RMSE:", final_stacking_rmse)
print("Final Stacking R²:", final_stacking_r2)

Final Stacking MAE: 10.628790952466511
Final Stacking RMSE: 14.01654542915614
Final Stacking R²: 0.9026841118329387


##### Our final model architecture

Original Extra Trees
        │
        ├───────┐
Tuned XGBoost  │
        │       ├──→ Ridge Meta-Model ──→ Demand Prediction
Tuned Random   │
Forest         │
        └───────┘

So at this point, Final Stacking Regression is our strongest model.

##### Adding Final Results to the Model Comparison

In [355]:
# Creating a DataFrame containing the performance of the tuned and final models.

additional_results = pd.DataFrame({
    'Model': [
        'Tuned Extra Trees Regression',
        'Tuned XGBoost Regression',
        'Tuned Random Forest Regression',
        'Final Stacking Regression'
    ],
    'MAE': [
        tuned_extra_trees_mae,
        tuned_xgboost_mae,
        tuned_random_forest_mae,
        final_stacking_mae
    ],
    'RMSE': [
        tuned_extra_trees_rmse,
        tuned_xgboost_rmse,
        tuned_random_forest_rmse,
        final_stacking_rmse
    ],
    'R2': [
        tuned_extra_trees_r2,
        tuned_xgboost_r2,
        tuned_random_forest_r2,
        final_stacking_r2
    ]
})

additional_results

,Model,MAE,RMSE,R2
0,Tuned Extra Trees Regression,11.018017,14.809778,0.891358
1,Tuned XGBoost Regression,11.402505,14.890003,0.890177
2,Tuned Random Forest Regression,12.158307,16.001342,0.873172
3,Final Stacking Regression,10.628791,14.016545,0.902684


In [356]:
# Combining the original and additional model results into one comparison table.

final_model_results = pd.concat(
    [
        model_results,
        additional_results
    ],
    ignore_index=True
)

final_model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
1,Linear Regression,15.154549,20.365633,0.794554
2,Ridge Regression,15.154426,20.365599,0.794555
3,Lasso Regression,15.878328,21.387819,0.773413
4,Decision Tree Regression,17.284016,23.607680,0.723937
5,Random Forest Regression,12.218563,16.094710,0.871688
6,Gradient Boosting Regression,13.429953,17.480096,0.848648
7,XGBoost Regression,11.580912,15.131741,0.886583
8,Extra Trees Regression,10.876563,14.656385,0.893597
9,HistGradientBoosting Regression,11.783125,15.291003,0.884183


In [357]:
# Sorting all models from the lowest to the highest RMSE.

final_model_results = final_model_results.sort_values(
    by='RMSE',
    ascending=True
).reset_index(
    drop=True
)

final_model_results

,Model,MAE,RMSE,R2
0,Final Stacking Regression,10.628791,14.016545,0.902684
1,Stacking Regression,10.750887,14.194631,0.900196
2,Voting Regression,11.044994,14.545320,0.895203
3,Extra Trees Regression,10.876563,14.656385,0.893597
4,Tuned Extra Trees Regression,11.018017,14.809778,0.891358
5,Tuned XGBoost Regression,11.402505,14.890003,0.890177
6,XGBoost Regression,11.580912,15.131741,0.886583
7,HistGradientBoosting Regression,11.783125,15.291003,0.884183
8,Tuned Random Forest Regression,12.158307,16.001342,0.873172
9,Random Forest Regression,12.218563,16.094710,0.871688


#### Identifying the Best Model

In [358]:
# Displaying the best-performing model based on the lowest RMSE.

best_model_result = final_model_results.iloc[0]

best_model_result

Model    Final Stacking Regression
MAE                      10.628791
RMSE                     14.016545
R2                        0.902684
Name: 0, dtype: object

In [359]:
# Assigning the best-performing trained model as the final model for further evaluation.

final_model = final_stacking

#### Saving the Final Model

In [360]:
# Importing joblib for saving and loading trained machine learning objects.

import joblib

In [361]:
# Saving the selected final stacking model for future use.

joblib.dump(
    final_model,
    '../models/final_stacking_model.pkl'
)

['../models/final_stacking_model.pkl']

##### Verifying the Saved Model and Preprocessor

In [362]:
# Loading the saved final model and preprocessor to verify that they were stored successfully.

loaded_final_model = joblib.load(
    '../models/final_stacking_model.pkl'
)

In [363]:
# Displaying the loaded objects to verify that the model and preprocessor were loaded successfully.

print("Final model loaded successfully:")
print(type(loaded_final_model))

Final model loaded successfully:
<class 'sklearn.ensemble._stacking.StackingRegressor'>


In [364]:
# Saving the fitted preprocessing transformer again.

joblib.dump(
    preprocessor,
    '../models/preprocessor.pkl'
)

['../models/preprocessor.pkl']

In [365]:
import os

print(
    os.path.exists(
        '../models/preprocessor.pkl'
    )
)

True


In [366]:
loaded_preprocessor = joblib.load(
    '../models/preprocessor.pkl'
)

print(type(loaded_preprocessor))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>
